# Modeling

Treina e compara vários modelos de classificação usando `data_preparation.py` e `data_modeling.py`, e guarda o melhor em `models/`.

In [12]:
import sys
sys.path.append("../src")  # módulos .py ficam em src/, notebooks em notebooks/

import data_preparation as dp
import data_modelling as dm

## 1. Preparar os dados

Reaproveita a fase anterior.

In [13]:
DATASET_NAME = "malware"
DATA_PATH = "../dataset/raw/Malware_and_benign_recognition.csv"

X_train, X_test, y_train, y_test = dp.run_data_preparation(DATA_PATH, target="Malicious")

Dimensões do dataset: (1156, 27)

--- Valores nulos ---
File                       0
SizeOfCode                 0
SizeOfInitializedData      0
SizeOfUninitializedData    0
AddressOfEntryPoint        0
BaseOfCode                 0
BaseOfData                 0
ImageBase                  0
NumberOfSections           0
DllCharacteristics         0
SizeOfImage                0
SizeOfHeaders              0
Characteristics            0
Subsystem                  0
SectionCount               0
AvgSectionEntropy          0
MaxSectionEntropy          0
MinSectionEntropy          0
TextSectionSize            0
DataSectionSize            0
SuspiciousSectionNames     0
ImportedDLLs               0
ImportedFunctions          0
HasExportTable             0
ExportedFunctions          0
OverlaySize                0
Malicious                  0
dtype: int64

--- Duplicados ---
Número de duplicados: 0

--- Distribuição das classes (Malicious) ---
Malicious
0    50.4
1    49.6
Name: proportion, dtype: flo

## 2. Comparar modelos com validação cruzada

Testa Regressão Logística, kNN, SVM, Naive Bayes, MLP, Árvore de Decisão, Random Forest e Gradient Boosting, com `StratifiedKFold` (normalização aplicada só aos modelos que precisam).

In [14]:
results_df = dm.evaluate_models(X_train, y_train, scoring="f1_macro")
results_df

Regressão Logística    | f1_macro = 0.9794 ± 0.0063
kNN                    | f1_macro = 0.9849 ± 0.0040
SVM                    | f1_macro = 0.9849 ± 0.0040
Naive Bayes            | f1_macro = 0.9371 ± 0.0121
MLP                    | f1_macro = 0.9881 ± 0.0040
Árvore de Decisão      | f1_macro = 0.9784 ± 0.0059
Random Forest          | f1_macro = 0.9935 ± 0.0041
Gradient Boosting      | f1_macro = 0.9924 ± 0.0055


,modelo,media,desvio_padrao
0,Random Forest,0.993502,0.004058
1,Gradient Boosting,0.992420,0.005520
2,MLP,0.988096,0.004043
3,kNN,0.984852,0.004034
4,SVM,0.984852,0.004034
5,Regressão Logística,0.979441,0.006292
6,Árvore de Decisão,0.978364,0.005900
7,Naive Bayes,0.937117,0.012146


In [15]:
dm.plot_model_comparison(results_df, scoring="f1_macro")

Gráfico guardado em: comparacao_modelos.png


## 3. Escolher e treinar o modelo campeão

In [16]:
best_name = dm.select_best_model(results_df)
best_pipe = dm.train_final_model(best_name, X_train, y_train)


Melhor modelo: Random Forest


## 4. Avaliar no conjunto de teste

In [17]:
dm.evaluate_on_test(best_pipe, X_test, y_test)


--- Classification report (teste) ---
              precision    recall  f1-score   support

           0       0.99      0.94      0.96       117
           1       0.94      0.99      0.97       115

    accuracy                           0.97       232
   macro avg       0.97      0.97      0.97       232
weighted avg       0.97      0.97      0.97       232

Gráfico guardado em: matriz_confusao.png


## 5. Guardar o modelo campeão

Fica em `../models/<dataset>_<modelo>.joblib` — uma pasta `models/` ao lado da pasta do código (ex.: `src/`), já existente no repositório — pronto a ser carregado pelo sistema multiagente.

In [18]:
dm.save_model(best_pipe, best_name, dataset_name=DATASET_NAME, output_dir="../models")


Modelo guardado em: ../models\malware_random_forest.joblib


'../models\\malware_random_forest.joblib'

## Alternativa: correr tudo de uma vez

In [19]:
# best_pipe = dm.run_modeling(X_train, X_test, y_train, y_test, dataset_name=DATASET_NAME)